In [ ]:
#%%
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go

np.random.seed(42)


In [ ]:
n_filas = 25

df = pd.DataFrame({
    'X': np.random.uniform(0, 10, n_filas),
    'Y': np.random.uniform(0, 10, n_filas),
    'Z': np.random.uniform(0, 10, n_filas)
})

df_origen = pd.DataFrame({'X': [0], 'Y': [0], 'Z': [0]})
df = pd.concat([df_origen, df], ignore_index=True)

# --- Crear gráfica ---
fig = go.Figure()

# Puntos normales (todos excepto el origen)
fig.add_trace(go.Scatter3d(
    x=df['X'][1:],
    y=df['Y'][1:],
    z=df['Z'][1:],
    mode='markers',
    marker=dict(
        size=6,
        color='darkblue',
        opacity=0.7,
        line=dict(width=0.5, color='darkblue')
    ),
    name='Puntos'
))

# Origen (0,0,0) en rojo
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    marker=dict(
        size=6,
        color='red',
        opacity=1,
        line=dict(width=2, color='darkred')
    ),
    name='Origen (0,0,0)'
))

# --- Personalizar layout ---
fig.update_layout(
    title='Nube de puntos 3D - Origen en rojo',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        xaxis=dict(range=[-1, 11]),
        yaxis=dict(range=[-1, 11]),
        zaxis=dict(range=[-1, 11])
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [ ]:
# --- PCA ---
scaler = StandardScaler()
Z = scaler.fit_transform(df)
Z = pd.DataFrame(Z, columns=df.columns)

# --- Graficar Z (datos estandarizados) ---
fig = go.Figure()

# Puntos normales de Z (todos excepto el origen)
fig.add_trace(go.Scatter3d(
    x=Z['X'][1:],
    y=Z['Y'][1:],
    z=Z['Z'][1:],
    mode='markers',
    marker=dict(
        size=6,
        color='darkblue',
        opacity=0.7,
        line=dict(width=0.5, color='darkblue')
    ),
    name='Puntos estandarizados'
))

# Origen (0,0,0) en rojo
fig.add_trace(go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode='markers',
    marker=dict(
        size=6,
        color='red',
        opacity=1,
        line=dict(width=2, color='darkred')
    ),
    name='Origen (0,0,0)'
))

# --- Personalizar layout ---
fig.update_layout(
    title='Nube de puntos 3D - Datos estandarizados (Z)<br>(Media=0, Desviación=1)',
    scene=dict(
        xaxis_title='X estandarizada',
        yaxis_title='Y estandarizada',
        zaxis_title='Z estandarizada',
        xaxis=dict(range=[-3, 3]),
        yaxis=dict(range=[-3, 3]),
        zaxis=dict(range=[-3, 3])
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()






In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(Z)

pc_vectors = pca.components_

# --- Gráfico 3D ---
fig = go.Figure()

# 1. Puntos originales en R³
# Separamos el origen (índice 0) del resto para poder darle color y tamaño diferente
fig.add_trace(go.Scatter3d(
    x=df['X'].iloc[1:],      # todos excepto el origen
    y=df['Y'].iloc[1:],
    z=df['Z'].iloc[1:],
    mode='markers',
    name='Puntos originales',
    marker=dict(size=7, color='royalblue', opacity=0.85),
    hovertemplate='X: %{x:.2f}<br>Y: %{y:.2f}<br>Z: %{z:.2f}<extra></extra>'
))

# Punto origen (0,0,0) en rojo y tamaño 6
fig.add_trace(go.Scatter3d(
    x=[df['X'].iloc[0]],
    y=[df['Y'].iloc[0]],
    z=[df['Z'].iloc[0]],
    mode='markers',
    name='Origen (0,0,0)',
    marker=dict(size=6, color='red', opacity=1.0, symbol='circle'),
    hovertemplate='Origen (0,0,0)<extra></extra>'
))

# 2. Plano PC1-PC2 (verde claro semi-transparente)
scale_plane = 9.0
v1 = pc_vectors[0] * scale_plane
v2 = pc_vectors[1] * scale_plane

u, v = np.meshgrid(np.linspace(-1, 1, 25), np.linspace(-1, 1, 25))
xx = u.flatten() * v1[0] + v.flatten() * v2[0]
yy = u.flatten() * v1[1] + v.flatten() * v2[1]
zz = u.flatten() * v1[2] + v.flatten() * v2[2]

fig.add_trace(go.Mesh3d(
    x=xx, y=yy, z=zz,
    opacity=0.22,
    color='palegreen',
    name='Plano PC1 + PC2',
    hoverinfo='skip'
))

# 3. Flechas de PC1 y PC2
colors = ['crimson', 'darkorange']
labels = ['PC1', 'PC2']

for i in range(2):
    vec = pc_vectors[i] * scale_plane * 0.85
    fig.add_trace(go.Scatter3d(
        x=[0, vec[0]],
        y=[0, vec[1]],
        z=[0, vec[2]],
        mode='lines+text',
        name=labels[i],
        line=dict(color=colors[i], width=7),
        text=['', labels[i]],
        textposition='middle right',
        textfont=dict(color=colors[i], size=14),
        hovertemplate=f'{labels[i]}<extra></extra>'
    ))

# 4. Puntos proyectados en el plano (darkgreen rellenos)
projections = X_pca @ pc_vectors

fig.add_trace(go.Scatter3d(
    x=projections[:,0],
    y=projections[:,1],
    z=projections[:,2],
    mode='markers',
    name='Proyecciones en el plano',
    marker=dict(
        size=6,
        color='darkgreen',
        opacity=0.9,
        symbol='circle'
    ),
    hovertemplate='Proyección en plano<br>X: %{x:.2f}<br>Y: %{y:.2f}<br>Z: %{z:.2f}<extra></extra>'
))

# Layout final
explained = pca.explained_variance_ratio_ * 100

fig.update_layout(
    title='Visualización en R³ — Puntos originales + Plano PC1-PC2<br>'
          f'Proyecciones marcadas en darkgreen',
    scene=dict(
        xaxis_title='Eje X',
        yaxis_title='Eje Y',
        zaxis_title='Eje Z',
        aspectmode='cube'
    ),
    width=1050,
    height=850,
    template='plotly_white',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.9)')
)

fig.show()